> **Niveau 🟢 facile — le code est écrit, complétez les `___`**

# Notebook 2 — De l'ADN du patient à l'hémoglobine en 3D

Le notebook 1 a traduit le gène *HBB* normal en bêta-globine. Ce notebook part de l'ADN du
patient, et pose trois questions :

1. quel gène est muté chez le patient, et quel acide aminé change ? (sections 1 à 3)
2. la chaîne β du patient a-t-elle la même forme que la chaîne normale ? (sections 4 à 6)
3. si oui, qu'est-ce que cet acide aminé change ? (sections 7 et 8)

Deux sources de structures :

| Source | Hémoglobine normale | Hémoglobine du patient |
|---|---|---|
| Prédiction d'**AlphaFold 3**, que vous lancez sur le serveur AlphaFold | `hb_normal` | `hb_patient` |
| Structure **expérimentale** (cristallographie aux rayons X, Protein Data Bank) | `2HHB` | `2HBS` |

`2HHB` et `2HBS` sont deux hémoglobines humaines sous forme désoxy (sans O₂) : l'hémoglobine
normale (HbA) et l'hémoglobine S (HbS), qui porte la mutation du patient.

**Mode d'emploi**
- `Maj + Entrée` exécute une cellule et passe à la suivante.
- Exécutez les cellules **dans l'ordre**, de haut en bas.
- Les cellules **✔️ Vérification** ne se modifient pas : elles affichent ✅ quand votre code est juste.
- Les vues 3D se manipulent à la souris : glisser pour tourner, molette pour zoomer.

## 0. Préparer les outils

Exécutez la cellule ci-dessous. Elle installe `py3Dmol`, une bibliothèque d'affichage 3D, et
définit les fonctions utilisées dans tout le notebook. Celles dont vous vous servirez :

| Fonction | Ce qu'elle renvoie |
|---|---|
| `telecharger(nom)` | le chemin du fichier `nom`, téléchargé si besoin (PDB, AlphaFold DB, ou copie de secours du cours) |
| `lire_structure(fichier)` | la liste des atomes d'un fichier `.pdb` ou `.cif` |
| `carbones_alpha(atomes, chaine)` | les carbones α d'une chaîne, dans un dictionnaire {numéro de résidu : atome} |
| `garder_chaines(atomes, chaines)` | les atomes des chaînes demandées, par exemple `"ABCD"` |
| `chaines_proteiques(atomes)` | la liste des chaînes formées d'acides aminés |

In [ ]:
#@title ▶️ Exécutez cette cellule pour préparer les outils (ne pas modifier)
# Installe py3Dmol (affichage 3D) et définit les fonctions utilisées dans le notebook.
!pip install -q py3Dmol

import math
import os
import urllib.request
import zipfile

import py3Dmol

# Copie de secours des fichiers du cours : le dépôt GitHub des notebooks
DEPOT = "https://raw.githubusercontent.com/cedricusureau/cours-bio-notebooks/main/data/"
# Prédictions du serveur AlphaFold préparées pour le cours (quand elles sont en ligne)
SECOURS = {"normale": "af3_hb_normal.cif", "patient": "af3_hb_patient.cif"}

# Code à trois lettres des 20 acides aminés → code à une lettre
UNE_LETTRE = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E",
    "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F",
    "PRO": "P", "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
}
# Atomes du squelette, communs aux vingt acides aminés (OXT : oxygène terminal de la chaîne)
SQUELETTE = ["N", "CA", "C", "O", "OXT"]
EAU = ["HOH", "WAT", "DOD"]

# Classe de chaque acide aminé, et sa couleur dans les vues 3D
CLASSE = {}
for lettre in "GAVLIMFWP":
    CLASSE[lettre] = "apolaire"
for lettre in "STCYNQ":
    CLASSE[lettre] = "polaire"
for lettre in "DE":
    CLASSE[lettre] = "acide"
for lettre in "KRH":
    CLASSE[lettre] = "basique"
COULEUR_CLASSE = {"apolaire": "#d97706", "polaire": "#0ea5e9", "acide": "#f43f5e",
                  "basique": "#2563eb"}
COULEUR_RESIDU = {}
for nom, lettre in UNE_LETTRE.items():
    COULEUR_RESIDU[nom] = COULEUR_CLASSE[CLASSE[lettre]]


def telecharger(nom, silencieux=False):
    """Rend le fichier `nom` disponible et renvoie son chemin (None en cas d'échec).
    Cherche d'abord une copie locale, puis télécharge : depuis la PDB (RCSB) pour une
    entrée PDB, depuis AlphaFold DB pour un modèle AF-..., et en dernier recours depuis
    la copie de secours du dépôt du cours."""
    for chemin in (nom, os.path.join("data", nom)):
        if os.path.exists(chemin):
            return chemin
    sources = []
    if len(nom) == 8 and nom.endswith((".pdb", ".cif")):
        sources.append("https://files.rcsb.org/download/" + nom)
    if nom.startswith("AF-"):
        sources.append("https://alphafold.ebi.ac.uk/files/" + nom)
    sources.append(DEPOT + nom)
    for url in sources:
        try:
            with urllib.request.urlopen(url, timeout=30) as reponse:
                contenu = reponse.read()
        except Exception:
            continue
        with open(nom, "wb") as f:
            f.write(contenu)
        if not silencieux:
            print(nom, ": téléchargé depuis", url.split("/")[2])
        return nom
    if not silencieux:
        print("❌", nom, ": téléchargement impossible (pas de réseau, ou fichier absent en ligne)")
    return None


def lire_structure(fichier):
    """Lit un fichier de structure .pdb ou .cif (mmCIF) et renvoie la liste de ses atomes.
    Chaque atome est un dictionnaire : chaine, residu (numéro), nom_residu, atome (nom),
    x, y, z (en Å) et b. Dans une prédiction AlphaFold, b contient le pLDDT.
    Premier modèle seulement ; molécules d'eau et atomes d'hydrogène ignorés."""
    if fichier is None:
        raise FileNotFoundError("aucun fichier à lire : voir le message de téléchargement plus haut")
    with open(fichier) as f:
        lignes = f.read().splitlines()
    if fichier.endswith(".cif"):
        return _lire_mmcif(lignes)
    return _lire_pdb(lignes)


def _lire_pdb(lignes):
    atomes = []
    for ligne in lignes:
        if ligne.startswith("ENDMDL"):
            break
        if not ligne.startswith(("ATOM", "HETATM")):
            continue
        nom_residu = ligne[17:20].strip()
        element = ligne[76:78].strip()
        if nom_residu in EAU or element in ("H", "D") or ligne[16] not in " A":
            continue
        atomes.append({
            "chaine": ligne[21],
            "residu": int(ligne[22:26]),
            "nom_residu": nom_residu,
            "atome": ligne[12:16].strip(),
            "x": float(ligne[30:38]),
            "y": float(ligne[38:46]),
            "z": float(ligne[46:54]),
            "b": float(ligne[60:66]),
        })
    return atomes


def _valeur(champs, *noms):
    """La première valeur renseignée parmi les colonnes `noms` ("." et "?" : non renseignée)."""
    for nom in noms:
        valeur = champs.get(nom)
        if valeur not in (None, ".", "?"):
            return valeur
    return None


def _lire_mmcif(lignes):
    """Boucle _atom_site d'un mmCIF : les colonnes sont repérées par leur nom."""
    colonnes = []
    atomes = []
    premier_modele = None
    for ligne in lignes:
        if ligne.startswith("_atom_site."):
            colonnes.append(ligne.split()[0][len("_atom_site."):])
            continue
        if not colonnes:
            continue
        if not ligne.startswith(("ATOM", "HETATM")):
            if atomes and (ligne.startswith("#") or ligne.startswith("loop_") or ligne.startswith("_")):
                break
            continue
        champs = dict(zip(colonnes, ligne.split()))
        modele = _valeur(champs, "pdbx_PDB_model_num")
        if premier_modele is None:
            premier_modele = modele
        if modele != premier_modele:
            continue
        nom_residu = _valeur(champs, "auth_comp_id", "label_comp_id")
        if nom_residu in EAU or _valeur(champs, "type_symbol") in ("H", "D"):
            continue
        if _valeur(champs, "label_alt_id") not in (None, "A"):
            continue
        numero = _valeur(champs, "auth_seq_id", "label_seq_id")
        atomes.append({
            "chaine": _valeur(champs, "auth_asym_id", "label_asym_id"),
            "residu": int(numero) if numero and numero.lstrip("-").isdigit() else 0,
            "nom_residu": nom_residu,
            "atome": _valeur(champs, "auth_atom_id", "label_atom_id").strip('"'),
            "x": float(champs["Cartn_x"]),
            "y": float(champs["Cartn_y"]),
            "z": float(champs["Cartn_z"]),
            "b": float(_valeur(champs, "B_iso_or_equiv") or 0),
        })
    return atomes


def carbones_alpha(atomes, chaine):
    """Les carbones α (atomes "CA") d'une chaîne : un dictionnaire {numéro de résidu: atome}."""
    ca = {}
    for atome in atomes:
        if atome["chaine"] == chaine and atome["atome"] == "CA" and atome["nom_residu"] in UNE_LETTRE:
            ca[atome["residu"]] = atome
    return ca


def garder_chaines(atomes, chaines):
    """Les atomes des chaînes demandées, par exemple garder_chaines(atomes, "ABCD")."""
    gardes = []
    for atome in atomes:
        if atome["chaine"] in chaines:
            gardes.append(atome)
    return gardes


def chaines_proteiques(atomes):
    """La liste triée des chaînes formées d'acides aminés (les hèmes sont exclus)."""
    chaines = []
    for atome in atomes:
        if atome["atome"] == "CA" and atome["nom_residu"] in UNE_LETTRE and atome["chaine"] not in chaines:
            chaines.append(atome["chaine"])
    return sorted(chaines)


def sequence_chaine(atomes, chaine):
    """La séquence d'une chaîne (code à une lettre), lue sur ses carbones α."""
    ca = carbones_alpha(atomes, chaine)
    sequence = ""
    for numero in sorted(ca):
        sequence = sequence + UNE_LETTRE[ca[numero]["nom_residu"]]
    return sequence


def resume_chaines(atomes):
    """Affiche chaque chaîne : son nombre de résidus et le début de sa séquence, ou ses ligands."""
    toutes = []
    for atome in atomes:
        if atome["chaine"] not in toutes:
            toutes.append(atome["chaine"])
    for chaine in sorted(toutes):
        sequence = sequence_chaine(atomes, chaine)
        if sequence:
            print(f"chaîne {chaine} : {len(sequence)} résidus   {sequence[:10]}...")
        else:
            noms = []
            for atome in atomes:
                if atome["chaine"] == chaine and atome["nom_residu"] not in noms:
                    noms.append(atome["nom_residu"])
            print(f"chaîne {chaine} : {', '.join(noms)}")


def chaine_beta(atomes):
    """La première chaîne β : 146 résidus, séquence commençant par VHLTP. None si aucune."""
    for chaine in chaines_proteiques(atomes):
        sequence = sequence_chaine(atomes, chaine)
        if len(sequence) == 146 and sequence.startswith("VHLTP"):
            return chaine
    return None


def chercher_predictions():
    """Cherche les deux prédictions du tétramère, normale et patient : d'abord vos fichiers
    déposés (.zip du serveur AlphaFold ou .cif), reconnus à l'acide aminé n°6 de leur chaîne β ;
    à défaut, les prédictions de secours du cours.
    Renvoie {"normale": fichier ou None, "patient": fichier ou None}."""
    # 1. extraire de chaque .zip déposé le modèle n°0, le mieux classé par AlphaFold
    for nom in sorted(os.listdir(".")):
        if nom.endswith(".zip"):
            try:
                with zipfile.ZipFile(nom) as archive:
                    for membre in archive.namelist():
                        if membre.endswith("model_0.cif"):
                            with open(os.path.basename(membre), "wb") as f:
                                f.write(archive.read(membre))
            except zipfile.BadZipFile:
                print("⚠️", nom, ": archive illisible, ignorée")
    # 2. les .cif déposés : glutamate en 6 → normale, valine en 6 → patient
    trouvees = {"normale": None, "patient": None}
    for nom in sorted(os.listdir(".")):
        if not nom.endswith(".cif") or nom in SECOURS.values():
            continue
        try:
            atomes = lire_structure(nom)
        except Exception:
            print("⚠️", nom, ": fichier illisible, ignoré")
            continue
        beta = chaine_beta(atomes)
        if beta is None:
            print("⚠️", nom, ": pas de chaîne β de 146 résidus, fichier ignoré")
            continue
        residu_6 = carbones_alpha(atomes, beta)[6]["nom_residu"]
        if residu_6 == "GLU" and trouvees["normale"] is None:
            trouvees["normale"] = nom
            print("prédiction normale  :", nom, "(votre fichier)")
        elif residu_6 == "VAL" and trouvees["patient"] is None:
            trouvees["patient"] = nom
            print("prédiction patient  :", nom, "(votre fichier)")
    # 3. à défaut, les prédictions de secours du cours
    for cle in ("normale", "patient"):
        if trouvees[cle] is None:
            trouvees[cle] = telecharger(SECOURS[cle], silencieux=True)
            if trouvees[cle]:
                print(f"prédiction {cle:8s} :", trouvees[cle], "(prédiction de secours du cours)")
            else:
                print(f"prédiction {cle:8s} : aucune — ni fichier déposé, ni prédiction de secours en ligne")
    return trouvees


def couleur_plddt(plddt):
    """La couleur d'un pLDDT, dans l'échelle d'AlphaFold."""
    if plddt > 90:
        return "#2563eb"
    if plddt > 70:
        return "#0ea5e9"
    if plddt > 50:
        return "#fde68a"
    return "#d97706"


def proche(valeur, attendu, tolerance):
    """Vrai si `valeur` est un nombre à moins de `tolerance` de `attendu` (vérifications)."""
    return isinstance(valeur, (int, float)) and abs(valeur - attendu) <= tolerance


def nouvel_atome(chaine, residu, nom, x=0.0, y=0.0, z=0.0, b=0.0, nom_residu="ALA"):
    """Un atome au format de lire_structure(), pour les exemples des vérifications."""
    return {"chaine": chaine, "residu": residu, "nom_residu": nom_residu, "atome": nom,
            "x": float(x), "y": float(y), "z": float(z), "b": float(b)}


# Style des étiquettes des vues 3D
ETIQUETTE = {"fontSize": 13, "fontColor": "#0f172a", "backgroundColor": "#ffffff",
             "backgroundOpacity": 0.85, "borderColor": "#0f172a", "borderThickness": 1}


def vue_3d(fichiers, largeur=900, hauteur=420, liees=True):
    """Prépare une vue py3Dmol d'un fichier, ou de plusieurs fichiers côte à côte (les vues
    tournent ensemble si liees=True). Rien n'est dessiné au départ : on ajoute ensuite des
    styles (setStyle, addSurface, addLabel...), puis on affiche avec vue.show()."""
    if isinstance(fichiers, str):
        fichiers = [fichiers]
    vue = py3Dmol.view(width=largeur, height=hauteur, viewergrid=(1, len(fichiers)), linked=liees)
    for k, fichier in enumerate(fichiers):
        with open(fichier) as f:
            lignes = f.readlines()
        if fichier.endswith(".pdb"):
            lignes = [ligne for ligne in lignes if ligne[17:20] not in EAU]
            vue.addModel("".join(lignes), "pdb", viewer=(0, k))
        else:
            vue.addModel("".join(lignes), "cif", viewer=(0, k))
    vue.setStyle({}, {})
    return vue


def tourner_vers(vue, atomes, chaine, residu, k=0):
    """Tourne la vue n°k (de gauche à droite, à partir de 0) pour placer le résidu (chaine,
    residu) face à l'écran : la direction du centre de la molécule vers le résidu devient
    l'axe de visée. À appeler après vue.zoomTo(), qui centre sur ces mêmes atomes."""
    molecule = []
    for atome in atomes:
        if atome["nom_residu"] in UNE_LETTRE:
            molecule.append(atome)
    cible = []
    for atome in molecule:
        if atome["chaine"] == chaine and atome["residu"] == residu:
            cible.append(atome)
    ecart = []
    for axe in ("x", "y", "z"):
        centre = sum(a[axe] for a in molecule) / len(molecule)
        ecart.append(sum(a[axe] for a in cible) / len(cible) - centre)
    x, y, z = ecart
    # rotation autour de x, qui annule la composante y, puis autour de y, qui annule x
    vue.rotate(math.degrees(math.atan2(-x, math.hypot(y, z))), "y", viewer=(0, k))
    vue.rotate(math.degrees(math.atan2(y, z)), "x", viewer=(0, k))


print("outils prêts ✅")

Un fichier de structure décrit une molécule **atome par atome** : pour chaque atome, la chaîne,
le numéro et le nom du résidu, le nom de l'atome et ses coordonnées x, y, z en ångströms
(1 Å = 10⁻¹⁰ m). La cellule suivante télécharge les deux structures expérimentales et montre
deux formes : le fichier tel qu'il est écrit, puis ce qu'en fait `lire_structure()` — une
**liste de dictionnaires**, un par atome. Les molécules d'eau du cristal ne sont pas gardées.

In [ ]:
hb_normale = lire_structure(telecharger("2HHB.pdb"))
hb_patient = lire_structure(telecharger("2HBS.pdb"))

# le fichier, tel qu'il est écrit sur le disque : ses 3 premières lignes ATOM
print("2HHB.pdb, ses 3 premières lignes ATOM :")
n = 0
with open(telecharger("2HHB.pdb")) as f:
    for ligne in f:
        if ligne.startswith("ATOM") and n < 3:
            print(ligne.rstrip())
            n = n + 1
print("...")
print()

# ce que Python en a fait : une liste de dictionnaires, un par atome
print("2HHB :", len(hb_normale), "atomes ; 2HBS :", len(hb_patient), "atomes")
print("hb_normale[0] =", hb_normale[0])

## 1. Quel gène est muté chez le patient ?

On dispose de la séquence codante de **quatre gènes exprimés dans le globule rouge**, chez un
sujet de référence et chez le patient :

| Gène | Protéine |
|---|---|
| `HBA1` | alpha-globine |
| `HBB` | bêta-globine |
| `HBD` | delta-globine |
| `HBG1` | gamma-globine (hémoglobine fœtale) |

Exécutez la cellule ci-dessous : elle crée deux fichiers FASTA — la référence et le patient —
et range leur contenu dans deux dictionnaires, `REFERENCE` et `PATIENT`. Elle redéfinit aussi
les trois fonctions du notebook 1 : `transcrire()`, `codons()` et `traduire()`.

In [ ]:
#@title ▶️ Exécutez cette cellule pour charger les séquences (ne pas modifier)
# Séquences codantes de référence (RefSeq) de quatre gènes exprimés dans le globule rouge,
# et les mêmes gènes séquencés chez le patient.

FASTA_REFERENCE = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

FASTA_PATIENT = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGTGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

with open("reference.fasta", "w") as f:
    f.write(FASTA_REFERENCE)
with open("patient.fasta", "w") as f:
    f.write(FASTA_PATIENT)


def lire_multifasta(nom_fichier):
    """Lit un fichier FASTA contenant plusieurs séquences.
    Renvoie un dictionnaire {nom du gène: séquence}."""
    sequences = {}
    nom = None
    with open(nom_fichier) as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne.startswith(">"):
                nom = ligne[1:].split()[0]
                sequences[nom] = ""
            elif ligne:
                sequences[nom] = sequences[nom] + ligne
    return sequences


CODE_GENETIQUE = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
    "UAU": "Y", "UAC": "Y", "UAA": "*", "UAG": "*",
    "UGU": "C", "UGC": "C", "UGA": "*", "UGG": "W",
    "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AUU": "I", "AUC": "I", "AUA": "I", "AUG": "M",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}


# Les trois fonctions écrites au notebook 1
def transcrire(adn):
    """Renvoie l'ARN messager : la séquence de adn avec chaque T remplacé par U."""
    return adn.replace("T", "U")


def codons(arn):
    """Découpe arn en triplets consécutifs à partir de l'index 0."""
    liste = []
    for i in range(0, len(arn), 3):
        liste.append(arn[i:i + 3])
    return liste


def traduire(arn):
    """Traduit arn en protéine (code à une lettre), jusqu'au premier codon stop exclu."""
    proteine = ""
    for codon in codons(arn):
        acide_amine = CODE_GENETIQUE[codon]
        if acide_amine == "*":
            break
        proteine = proteine + acide_amine
    return proteine


REFERENCE = lire_multifasta("reference.fasta")
PATIENT = lire_multifasta("patient.fasta")
print(len(REFERENCE), "gènes chargés ✅")

Un même fichier FASTA peut contenir plusieurs séquences à la suite — ici, les quatre gènes.
La fonction `lire_multifasta()` de la cellule précédente le transforme en **dictionnaire** : à
chaque nom de gène correspond sa séquence, en une seule chaîne de caractères.

In [ ]:
# le fichier, tel qu'il est écrit sur le disque (ses 5 premières lignes)
for ligne in FASTA_REFERENCE.splitlines()[:5]:
    print(ligne)
print("...")

print()

# ce que Python en a fait : un dictionnaire
print("les clés  :", list(REFERENCE))
print("REFERENCE['HBB'] :", REFERENCE["HBB"][:40], "...")
print("type      :", type(REFERENCE["HBB"]))

Les quatre gènes du patient ont été séquencés. Trois sont identiques à la référence, un seul
diffère — c'est celui-là qu'il faut trouver.

Pour parcourir un dictionnaire : `for cle in mon_dictionnaire:` donne ses clés, une à une. Deux
chaînes de caractères se comparent directement : `"ACGT" == "ACGT"` vaut `True`.

In [ ]:
gene_mute = None

for nom in REFERENCE:
    sequence_ref = REFERENCE[nom]
    # la séquence du même gène chez le patient
    sequence_patient = ___[nom]
    if sequence_ref == sequence_patient:
        print(nom, ": identique")
    else:
        print(nom, ": DIFFÉRENT")
        # retenir le nom de ce gène
        gene_mute = ___

print()
print("gène muté :", gene_mute)

# tout le reste du notebook portera sur ce gène
adn_wt = REFERENCE[gene_mute]
adn_patient = PATIENT[gene_mute]

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if gene_mute == "HBB":
    print("✅ le gène muté est HBB — la bêta-globine, une des deux chaînes de l'hémoglobine")
else:
    print("❌ gène trouvé :", gene_mute, "— reprenez la comparaison, gène par gène")

if adn_wt == REFERENCE["HBB"] and adn_patient == PATIENT["HBB"]:
    print("✅ adn_wt et adn_patient contiennent bien les séquences de ce gène")
else:
    print("❌ adn_wt et adn_patient doivent valoir REFERENCE[gene_mute] et PATIENT[gene_mute]")

## 2. Quelle base ?

À l'œil, 444 lettres, c'est trop. On écrit une fonction `trouver_mutations(ref, patient)` qui
parcourt les deux séquences position par position et renvoie la **liste des différences**,
chacune sous la forme `(index, base_ref, base_patient)`.

Exemple : `trouver_mutations("ACGT", "ACCT")` → `[(2, "G", "C")]`

⚠️ Python numérote à partir de **0** ; les biologistes numérotent les nucléotides à partir de **1**.
Le nucléotide n°1 est à l'index 0.

In [ ]:
def trouver_mutations(ref, patient):
    # une liste vide, qu'on remplira avec les différences trouvées
    mutations = []
    # parcourir tous les index i, de 0 à len(ref) exclu
    for i in range(___):
        # condition : la base de ref à l'index i est différente (!=) de celle du patient
        if ___:
            # ajouter le triplet (index, base de ref, base du patient)
            mutations.append((i, ref[i], ___))
    return mutations


mutations = trouver_mutations(adn_wt, adn_patient)
for index, base_ref, base_patient in mutations:
    # numéro du nucléotide pour un biologiste = index Python + 1
    numero = ___
    print(f"index Python {index} = nucléotide n°{numero} : {base_ref} → {base_patient}")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
essai = trouver_mutations("ACGT", "ACCT")
if essai == [(2, "G", "C")]:
    print("✅ sur l'exemple ACGT / ACCT :", essai)
else:
    print("❌ sur l'exemple ACGT / ACCT : attendu [(2, 'G', 'C')], obtenu", essai)

if trouver_mutations("AAAA", "AAAA") == []:
    print("✅ deux séquences identiques donnent bien une liste vide")
else:
    print("❌ deux séquences identiques doivent donner une liste vide")

if len(mutations) == 1:
    print("✅ une seule différence entre la référence et le patient :", mutations)
else:
    print("❌ attendu 1 différence, trouvé", len(mutations))

## 3. Quel codon, quel acide aminé ?

Le ribosome lit l'ARN trois bases à la fois, à partir du codon start : le nucléotide d'index
`i` appartient au codon d'index `i // 3` (division entière). Les fonctions du notebook 1 donnent
les codons des deux séquences : `codons(transcrire(adn))`.

In [ ]:
# index du nucléotide muté (section 2) : le premier élément du premier triplet
index_mutation = mutations[0][0]
# chaque codon contient 3 nucléotides : on divise l'index par 3 (division entière //)
index_codon = index_mutation // ___

# les codons des deux séquences : fonctions transcrire() et codons() du notebook 1
codons_wt = codons(transcrire(adn_wt))
codons_patient = codons(transcrire(___))

print(f"la mutation est dans le codon n°{index_codon + 1} (index Python {index_codon})")
# CODE_GENETIQUE[codon] donne l'acide aminé du codon
print("codon référence :", codons_wt[index_codon], "→", CODE_GENETIQUE[codons_wt[index_codon]])
print("codon patient   :", codons_patient[index_codon], "→", CODE_GENETIQUE[___])

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if index_codon == 6 and codons_wt[6] == "GAG" and codons_patient[6] == "GUG":
    print("✅ codon n°7 (index 6) :", codons_wt[6], "→", codons_patient[6])
else:
    print("❌ la mutation doit tomber dans le codon n°7, c'est-à-dire l'index 6 : GAG → GUG")

Une protéine, comme l'ADN, est une chaîne de caractères : la fonction `trouver_mutations()` de
la section 2 marche donc aussi sur les protéines.

Le dictionnaire `NOMS` donne, pour chaque lettre, l'abréviation à trois lettres et le nom.

In [ ]:
NOMS = {
    "A": ("Ala", "alanine"),
    "R": ("Arg", "arginine"),
    "N": ("Asn", "asparagine"),
    "D": ("Asp", "aspartate"),
    "C": ("Cys", "cystéine"),
    "Q": ("Gln", "glutamine"),
    "E": ("Glu", "glutamate"),
    "G": ("Gly", "glycine"),
    "H": ("His", "histidine"),
    "I": ("Ile", "isoleucine"),
    "L": ("Leu", "leucine"),
    "K": ("Lys", "lysine"),
    "M": ("Met", "méthionine"),
    "F": ("Phe", "phénylalanine"),
    "P": ("Pro", "proline"),
    "S": ("Ser", "sérine"),
    "T": ("Thr", "thréonine"),
    "W": ("Trp", "tryptophane"),
    "Y": ("Tyr", "tyrosine"),
    "V": ("Val", "valine"),
}

print(NOMS["E"], NOMS["V"])

In [ ]:
# les deux protéines : transcrire(), puis traduire()
proteine_wt = traduire(transcrire(adn_wt))
proteine_patient = traduire(transcrire(___))

# réutiliser trouver_mutations sur les deux protéines
differences = trouver_mutations(___, ___)

for index, aa_wt, aa_patient in differences:
    # numéro de l'acide aminé = index Python + 1
    numero = ___
    print(f"position {numero} : {aa_wt} → {aa_patient}")
    # NOMS[lettre] donne (abréviation, nom) : [1] pour le nom
    print(f"  {NOMS[aa_wt][1]} → {NOMS[___][1]}")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if len(proteine_wt) == 147 and len(proteine_patient) == 147:
    print("✅ 147 acides aminés de chaque côté")
else:
    print("❌ attendu 147 acides aminés, trouvé", len(proteine_wt), "et", len(proteine_patient))

if differences == [(6, "E", "V")]:
    print("✅ diagnostic :", NOMS["E"][0], "→", NOMS["V"][0],
          "en position 7 de la chaîne traduite")
else:
    print("❌ attendu une seule différence, (6, 'E', 'V'), et vous obtenez", differences)

| Niveau | Référence | Patient |
|---|---|---|
| Gène muté parmi les quatre | — | `HBB`, la bêta-globine |
| ADN, nucléotide n°20 | `A` | `T` |
| ARN, codon n°7 | `GAG` | `GUG` |
| Protéine, acide aminé n°7 | E — glutamate | V — valine |

Une seule base modifiée sur 444 remplace un acide aminé chargé (glutamate) par un acide
aminé apolaire (valine).

**Une simplification à connaître.** Ici, les trois autres gènes du patient sont strictement
identiques à la référence. Chez un individu réel, chacun porterait des dizaines de variants
sans conséquence : la difficulté n'est pas de trouver *une* différence, c'est de repérer
celle qui change la protéine.

## 4. Les chaînes de l'hémoglobine et le serveur AlphaFold

L'hémoglobine est un **tétramère** : deux chaînes α (gène `HBA1`) et deux chaînes β (gène
`HBB`). Chaque chaîne porte un **hème** : une molécule plane, la porphyrine, avec en son centre
un atome de fer qui fixe une molécule d'O₂.

Dans la cellule suivante, `[1:]` retire le premier acide aminé de chaque chaîne traduite : la
**méthionine initiale**, absente de la protéine mature.

In [ ]:
# traduire(transcrire(...)) donne la chaîne traduite ; [1:] retire son premier acide aminé,
# la méthionine
alpha = traduire(transcrire(REFERENCE["HBA1"]))[1:]
beta_normale = traduire(transcrire(REFERENCE["HBB"]))[1:]
beta_patient = traduire(transcrire(PATIENT["HBB"]))[1:]

print("chaîne α,", len(alpha), "acides aminés :")
print(alpha)
print()
print("chaîne β normale,", len(beta_normale), "acides aminés :")
print(beta_normale)
print()
print("chaîne β du patient,", len(beta_patient), "acides aminés :")
print(beta_patient)
print()
print("acide aminé n°6 (index Python 5) :", beta_normale[5], "chez la normale,",
      beta_patient[5], "chez le patient")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if len(alpha) == 141 and len(beta_normale) == 146 and len(beta_patient) == 146:
    print("✅ chaînes matures : 141 acides aminés pour α, 146 pour β")
else:
    print("❌ attendu 141 (α) et 146 (β) acides aminés, trouvé",
          len(alpha), len(beta_normale), len(beta_patient))

if beta_normale[5] == "E" and beta_patient[5] == "V":
    print("✅ position 6 : E (glutamate) dans la chaîne normale, V (valine) chez le patient")
else:
    print("❌ attendu E puis V en position 6, trouvé", beta_normale[5], "et", beta_patient[5])

Sans la méthionine initiale, le glutamate muté passe de la position 7 (section 3) à la
**position 6** : c'est la numérotation de la littérature (Glu6Val) et des fichiers PDB.

### Lancer les prédictions sur le serveur AlphaFold

AlphaFold 3 prédit la structure d'un assemblage de molécules — protéines, acides nucléiques,
petites molécules comme l'hème — à partir de la séquence de chaque chaîne et du code de chaque
petite molécule (`HEM` pour l'hème). On lui soumet deux tétramères de 2 chaînes α, 2 chaînes β et
4 hèmes : l'un avec la chaîne β normale, l'autre avec celle du patient.

La cellule ci-dessous écrit ces deux demandes au format JSON du serveur (`af3_hb_normal.json` et
`af3_hb_patient.json`) et, dans Colab, les télécharge sur votre ordinateur ; si le navigateur
demande l'autorisation de télécharger plusieurs fichiers, acceptez. Un fichier JSON a la forme de
listes et de dictionnaires Python.

In [ ]:
import json


def demande_alphafold(nom, beta):
    """Une demande (job) au format du serveur AlphaFold : 2 chaînes α, 2 chaînes β, 4 hèmes."""
    return [{
        "name": nom,
        "modelSeeds": [],
        "sequences": [
            {"proteinChain": {"sequence": alpha, "count": 2}},
            {"proteinChain": {"sequence": beta, "count": 2}},
            {"ligand": {"ligand": "CCD_HEM", "count": 4}},
        ],
        "dialect": "alphafoldserver",
        "version": 1,
    }]


for nom, beta in [("hb_normal", beta_normale), ("hb_patient", beta_patient)]:
    with open("af3_" + nom + ".json", "w") as f:
        json.dump(demande_alphafold(nom, beta), f, indent=2)
    print("écrit : af3_" + nom + ".json")

# la demande du patient, telle qu'elle est écrite dans le fichier
print()
print(json.dumps(demande_alphafold("hb_patient", beta_patient), indent=2))

# dans Colab : télécharger les deux fichiers sur votre ordinateur
try:
    from google.colab import files
    files.download("af3_hb_normal.json")
    files.download("af3_hb_patient.json")
except ImportError:
    print("(hors de Colab : les deux fichiers sont dans le dossier du notebook)")

**Sur [alphafoldserver.com](https://alphafoldserver.com)**, dans un nouvel onglet :

1. connectez-vous avec votre compte Google (à la première connexion, acceptez les conditions
   d'utilisation) ;
2. cliquez sur **Upload JSON** et choisissez `af3_hb_normal.json` (dans vos téléchargements) ;
3. ouvrez le job importé et vérifiez-le : deux entités *Protein* de 2 copies chacune, une entité
   *Ligand* `HEM` de 4 copies ;
4. cliquez sur **Continue and preview job**, puis sur **Confirm and submit job** ;
5. recommencez avec `af3_hb_patient.json`.

Sans les fichiers JSON, on peut remplir le formulaire à la main, avec **Add entity** : *Protein*,
séquence α, 2 copies ; *Protein*, séquence β, 2 copies ; *Ligand*, `HEM`, 4 copies. Les séquences
à coller sont affichées plus haut. Les deux fichiers JSON sont aussi sur la
[page d'accueil des notebooks](https://cedricusureau.github.io/cours-bio-notebooks/).

Chaque prédiction prend quelques minutes : **continuez le notebook pendant ce temps**. Quand une
prédiction est terminée, ouvrez-la depuis l'historique des jobs et cliquez sur **Download** : vous
obtenez un fichier `.zip`, à déposer avec la cellule 📤 ci-dessous.

Sans compte, ou si le serveur est saturé, passez à la section 5 : le notebook utilise alors des
prédictions de secours.

### 📤 Déposer vos prédictions

À exécuter **quand vos prédictions sont terminées** — vous pouvez y revenir plus tard. Déposez le
ou les `.zip` téléchargés depuis le serveur, ou les fichiers `.cif` qu'ils contiennent. Le notebook
reconnaît la prédiction normale et celle du patient à l'acide aminé n°6 de leur chaîne β.

In [ ]:
#@title 📤 Déposer vos prédictions (fichiers .zip ou .cif du serveur AlphaFold)
try:
    from google.colab import files
    deposes = files.upload()
    print(len(deposes), "fichier(s) déposé(s) :", ", ".join(deposes))
except ImportError:
    print("Hors de Colab : copiez les .zip ou .cif du serveur AlphaFold dans le dossier du notebook.")
except Exception as erreur:
    print("Dépôt interrompu :", erreur)

## 5. Lire une prédiction : la confiance (pLDDT)

Pour chaque résidu, AlphaFold calcule un score de confiance, le **pLDDT**, de 0 à 100 :

| pLDDT | Confiance |
|---|---|
| > 90 | très fiable |
| 70 – 90 | fiable |
| 50 – 70 | faible |
| < 50 | très faible, souvent une région désordonnée |

Dans les fichiers produits par AlphaFold, le pLDDT occupe la colonne du **facteur B**, qui mesure
en cristallographie l'agitation des atomes autour de leur position : c'est `atome["b"]`.

La cellule suivante choisit la prédiction à lire : la vôtre si vous l'avez déposée ; sinon une
prédiction de secours préparée pour le cours ; à défaut, le modèle de la chaîne β seule publié
dans **AlphaFold DB**, la base des structures prédites par AlphaFold 2 (sans hème).

In [ ]:
predictions = chercher_predictions()

if predictions["normale"]:
    fichier_prediction = predictions["normale"]
else:
    print()
    print("→ Pas de prédiction du tétramère : on lit le modèle AlphaFold DB de la chaîne β seule.")
    fichier_prediction = telecharger("AF-P68871-F1-model_v6.pdb")

prediction = lire_structure(fichier_prediction)
print()
print("fichier lu :", fichier_prediction)
resume_chaines(prediction)

Le modèle AlphaFold DB porte sur la séquence de la chaîne β dans UniProt (P68871), méthionine
initiale comprise : 147 résidus, et le glutamate 6 y porte le numéro 7. Les prédictions du serveur,
faites sur les chaînes matures, suivent la numérotation des fichiers PDB.

**Exercice.** Écrivez `plddt_moyen(atomes, chaine)` : la moyenne du pLDDT sur les **carbones α**
de la chaîne `chaine`. Le carbone α est le carbone central de chaque acide aminé, celui qui porte
la chaîne latérale ; dans les fichiers, c'est l'atome nommé `"CA"`. Chaque résidu en a un seul :
chaque résidu compte une fois.

Affichez ensuite le pLDDT moyen de chaque chaîne de la prédiction. `chaines_proteiques(prediction)`
donne la liste de ses chaînes protéiques, sans les hèmes.

In [ ]:
def plddt_moyen(atomes, chaine):
    # la somme des pLDDT, et le nombre de carbones α rencontrés
    total = 0
    nombre = 0
    for atome in atomes:
        # garder les atomes de la chaîne demandée ET nommés "CA"
        if atome["chaine"] == ___ and atome["atome"] == ___:
            # le pLDDT est rangé dans le champ "b"
            total = total + atome[___]
            nombre = nombre + 1
    # la moyenne : la somme divisée par le nombre
    return ___


# une ligne par chaîne protéique de la prédiction
for chaine in chaines_proteiques(prediction):
    print("chaîne", chaine, ": pLDDT moyen", round(plddt_moyen(___, chaine), 1))

In [ ]:
# ✔️ Vérification — exécutez sans modifier
exemple = [
    nouvel_atome("A", 1, "CA", b=90),
    nouvel_atome("A", 1, "CB", b=10),
    nouvel_atome("A", 2, "CA", b=70),
    nouvel_atome("B", 1, "CA", b=20),
]
resultat = plddt_moyen(exemple, "A")
if proche(resultat, 80, 1e-9):
    print("✅ sur un exemple de 4 atomes : 80.0 — seuls les carbones α de la chaîne A comptent")
else:
    print("❌ sur l'exemple : attendu 80.0 (moyenne de 90 et 70), obtenu", resultat)

modele_afdb = lire_structure(telecharger("AF-P68871-F1-model_v6.pdb"))
resultat = plddt_moyen(modele_afdb, "A")
if proche(resultat, 97.18, 0.01):
    print("✅ pLDDT moyen du modèle AlphaFold DB de la chaîne β :", round(resultat, 1), "— très fiable")
else:
    print("❌ modèle AlphaFold DB de la chaîne β : attendu 97.18, obtenu", resultat)

La même prédiction en 3D. À gauche, les chaînes α en gris clair, les chaînes β en gris foncé,
les hèmes en bâtonnets (le modèle AlphaFold DB n'a qu'une chaîne β, sans hème). À droite, chaque
résidu dans la couleur de son pLDDT : bleu foncé > 90, bleu clair 70 – 90, jaune 50 – 70,
orange < 50.

In [ ]:
#@title ▶️ Vue 3D de la prédiction : les chaînes (à gauche), le pLDDT (à droite)
vue = vue_3d([fichier_prediction, fichier_prediction])
for chaine in chaines_proteiques(prediction):
    ca = carbones_alpha(prediction, chaine)
    # à gauche : chaînes α (141 résidus) en gris clair, chaînes β en gris foncé
    if len(ca) == 141:
        couleur = "#94a3b8"
    else:
        couleur = "#334155"
    vue.setStyle({"chain": chaine}, {"cartoon": {"color": couleur}}, viewer=(0, 0))
    # à droite : chaque résidu dans la couleur du pLDDT de son carbone α
    for numero, atome in ca.items():
        vue.setStyle({"chain": chaine, "resi": numero},
                     {"cartoon": {"color": couleur_plddt(atome["b"])}}, viewer=(0, 1))
# les hèmes, en bâtonnets
vue.setStyle({"resn": "HEM"}, {"stick": {"colorscheme": "redCarbon", "radius": 0.25}})
vue.zoomTo()
vue.show()

Dans le modèle AlphaFold DB de la chaîne β, 143 résidus sur 147 ont un pLDDT
supérieur à 90 ; les quatre autres sont aux extrémités de la chaîne (n° 1, 2, 3 et 147). Le
repliement de la globine est prédit avec une confiance très élevée.

## 6. Normale contre patient : même repliement ?

Deux structures de même repliement ont, aux imprécisions de mesure près, les mêmes **distances
internes** : la distance entre les carbones α de deux résidus donnés est la même dans l'une et
dans l'autre. Comparer les distances plutôt que les coordonnées dispense de superposer les deux
structures : une distance ne dépend ni de la position ni de l'orientation de la molécule dans le
fichier.

### La distance entre deux atomes

Pour deux atomes de coordonnées (x₁, y₁, z₁) et (x₂, y₂, z₂) :

d = √( (x₁ − x₂)² + (y₁ − y₂)² + (z₁ − z₂)² )

En Python, la puissance s'écrit `**`, et la racine carrée `** 0.5`.

Exemple : deux atomes placés en (0, 0, 0) et (1, 2, 2) sont à √(1 + 4 + 4) = 3 Å l'un de l'autre.

Écrivez `distance(atome1, atome2)`.

In [ ]:
def distance(atome1, atome2):
    # l'écart entre les deux atomes, sur chacun des trois axes
    dx = atome1["x"] - atome2["x"]
    dy = atome1[___] - atome2[___]
    dz = ___
    # la racine carrée de dx² + dy² + dz² : la racine s'écrit ** 0.5
    return (dx ** 2 + dy ** 2 + dz ** 2) ** ___


# carbones_alpha() range les carbones α d'une chaîne par numéro de résidu
ca = carbones_alpha(hb_normale, "B")
print("CA 1 – CA 2   :", round(distance(ca[1], ca[2]), 2), "Å")
# entre le premier et le dernier résidu de la chaîne β (n°146)
print("CA 1 – CA 146 :", round(distance(ca[1], ___), 2), "Å")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
atome_1 = nouvel_atome("A", 1, "CA", 0, 0, 0)
atome_2 = nouvel_atome("A", 2, "CA", 1, 2, 2)
resultat = distance(atome_1, atome_2)
if proche(resultat, 3, 1e-9) and proche(distance(atome_2, atome_1), 3, 1e-9):
    print("✅ atomes en (0, 0, 0) et (1, 2, 2) : 3.0 Å")
else:
    print("❌ atomes en (0, 0, 0) et (1, 2, 2) : attendu 3.0, obtenu", resultat)

ca = carbones_alpha(hb_normale, "B")
resultat = distance(ca[1], ca[2])
if proche(resultat, 3.72, 0.005):
    print("✅ carbones α des résidus 1 et 2 de la chaîne β :", round(resultat, 2), "Å")
    total = 0
    for numero in range(1, 146):
        total = total + distance(ca[numero], ca[numero + 1])
    print("   sur toute la chaîne, deux carbones α consécutifs sont en moyenne à", round(total / 145, 2),
          "Å : la longueur d'un maillon, fixée par la géométrie de la liaison peptidique")
else:
    print("❌ carbones α des résidus 1 et 2 de la chaîne B : attendu 3.72 Å, obtenu", resultat)

### L'écart de repliement

Écrivez `ecart_repliement(atomes1, chaine1, atomes2, chaine2)`, qui compare la chaîne `chaine1`
de la structure `atomes1` à la chaîne `chaine2` de la structure `atomes2` :

1. les carbones α de chaque chaîne : `carbones_alpha()` ;
2. les numéros de résidus présents dans les deux chaînes ;
3. pour chaque paire de ces résidus, la distance entre leurs carbones α dans la première
   structure (d₁) et dans la seconde (d₂) ;
4. la racine carrée de la moyenne des (d₁ − d₂)², sur toutes les paires.

Ce calcul s'appelle le **dRMSD** (*distance root-mean-square deviation*). Il s'exprime en
ångströms : 0 pour deux structures de même forme, d'autant plus grand que les formes diffèrent.

Exemple : trois résidus alignés à 3 Å d'intervalle, puis les mêmes avec le troisième éloigné de
3 Å dans le même axe. Les distances passent de (3, 6, 3) à (3, 9, 6), et le
dRMSD vaut √((0² + 3² + 3²) / 3) = √6 ≈ 2,45 Å.

Appliquez-la aux chaînes β, `"B"`, de la structure normale et de celle du patient.

In [ ]:
def ecart_repliement(atomes1, chaine1, atomes2, chaine2):
    # les carbones α de chaque chaîne, rangés par numéro de résidu : {numéro: atome}
    ca1 = carbones_alpha(atomes1, chaine1)
    ca2 = carbones_alpha(atomes2, ___)
    # les numéros de résidus présents dans les deux chaînes
    communs = []
    for numero in ca1:
        if numero in ___:
            communs.append(numero)
    somme = 0
    paires = 0
    # chaque paire (i, j) une seule fois : j commence juste après i
    for i in range(len(communs)):
        for j in range(___, len(communs)):
            # la distance entre les deux mêmes résidus, dans chaque structure
            d1 = distance(ca1[communs[i]], ca1[communs[j]])
            d2 = distance(ca2[communs[i]], ca2[___])
            somme = somme + (d1 - d2) ** 2
            paires = paires + 1
    # la racine carrée de la moyenne
    return (___ / ___) ** 0.5


ecart = ecart_repliement(hb_normale, "B", hb_patient, "B")
print("chaîne β normale (2HHB) contre chaîne β du patient (2HBS) :", round(ecart, 2), "Å")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
droite = [nouvel_atome("A", 1, "CA", 0, 0, 0), nouvel_atome("A", 2, "CA", 3, 0, 0),
          nouvel_atome("A", 3, "CA", 6, 0, 0)]
etiree = [nouvel_atome("B", 1, "CA", 0, 0, 0), nouvel_atome("B", 2, "CA", 3, 0, 0),
          nouvel_atome("B", 3, "CA", 9, 0, 0), nouvel_atome("B", 4, "CA", 20, 0, 0)]
resultat = ecart_repliement(droite, "A", etiree, "B")
if proche(resultat, 6 ** 0.5, 1e-9):
    print("✅ sur l'exemple : √6 ≈ 2.449 Å (le résidu 4, absent de la première structure, est ignoré)")
else:
    print("❌ sur l'exemple : attendu √6 ≈ 2.449 (3 paires ; distances (3, 6, 3) contre (3, 9, 6)),",
          "obtenu", resultat)

resultat = ecart_repliement(hb_normale, "B", hb_normale, "B")
if proche(resultat, 0, 1e-9):
    print("✅ une chaîne comparée à elle-même : 0.0 Å")
else:
    print("❌ une chaîne comparée à elle-même doit donner 0, obtenu", resultat)

resultat = ecart_repliement(hb_normale, "B", hb_patient, "B")
if proche(resultat, 0.257, 0.002):
    print("✅ chaîne β normale contre chaîne β du patient :", round(resultat, 2), "Å")
else:
    print("❌ chaîne β normale contre chaîne β du patient : attendu 0.257 Å, obtenu", resultat)

### Petit ou grand ? Deux points de comparaison

Un écart se lit par comparaison :

- **deux copies de la même chaîne** : les chaînes β `B` et `D` de 2HHB ont la même séquence et
  sont mesurées dans le même cristal ; leur écart vient de l'expérience elle-même (précision de
  la mesure, environnement de chaque chaîne dans le cristal) ;
- **deux chaînes différentes au repliement voisin** : une chaîne α et une chaîne β, identiques en
  64 positions sur les 139 qu'on peut apparier. Leurs numérotations diffèrent :
  la cellule ramène d'abord la chaîne α dans la numérotation de β, d'après l'alignement de leurs
  séquences.

In [ ]:
print("β normale (2HHB, B) contre β du patient (2HBS, B) :",
      round(ecart_repliement(hb_normale, "B", hb_patient, "B"), 2), "Å")
print("deux copies de β dans 2HHB (chaîne B contre D)    :",
      round(ecart_repliement(hb_normale, "B", hb_normale, "D"), 2), "Å")

# La chaîne α (A) renumérotée dans la numérotation de β, d'après l'alignement des deux
# séquences : le résidu α n devient le résidu β n + décalage, pour n de debut à fin.
# Sans homologue : les résidus α 18 et 19 ; les résidus β 2, 46 et 50 à 54.
ALIGNEMENT_ALPHA_BETA = [(1, 1, 0), (2, 17, 1), (20, 46, -1), (47, 49, 0), (50, 141, 5)]

alpha_numerotee_beta = []
for atome in hb_normale:
    if atome["chaine"] == "A":
        for debut, fin, decalage in ALIGNEMENT_ALPHA_BETA:
            if debut <= atome["residu"] <= fin:
                copie = dict(atome)
                copie["residu"] = atome["residu"] + decalage
                alpha_numerotee_beta.append(copie)

print("chaîne α contre chaîne β dans 2HHB (A contre B)   :",
      round(ecart_repliement(alpha_numerotee_beta, "A", hb_normale, "B"), 2), "Å")

L'écart entre la chaîne β normale et celle du patient (0,26 Å) est du même ordre que
l'écart entre deux copies de la même chaîne (0,19 Å), et environ quatre fois plus
petit que l'écart entre les chaînes α et β (1,12 Å) : **la chaîne β du patient a le même
repliement que la chaîne normale.**

### Et les prédictions d'AlphaFold ?

Même calcul sur les deux prédictions du serveur, chaîne β contre chaîne β. Si elles ne sont pas
encore disponibles, revenez à cette cellule plus tard.

In [ ]:
predictions = chercher_predictions()
print()

if predictions["normale"] and predictions["patient"]:
    af_normale = lire_structure(predictions["normale"])
    af_patient = lire_structure(predictions["patient"])
    beta_1 = chaine_beta(af_normale)
    beta_2 = chaine_beta(af_patient)
    ecart_af = ecart_repliement(af_normale, beta_1, af_patient, beta_2)
    print(f"prédiction normale (chaîne {beta_1}) contre prédiction du patient (chaîne {beta_2}) :",
          round(ecart_af, 2), "Å")
else:
    print("Les deux prédictions du tétramère ne sont pas disponibles.")
    print("Revenez à cette cellule quand vous les aurez déposées (section 4).")

AlphaFold n'a pas été entraîné à prédire l'effet d'une mutation ponctuelle : deux prédictions de
même forme ne suffiraient pas, seules, à conclure. C'est la comparaison des structures
expérimentales, plus haut, qui établit que la mutation ne modifie pas le repliement.

## 7. Où est le résidu 6 ?

Le repliement ne change pas. Reste à situer, dans ce repliement, le résidu qui change.

Dans une protéine repliée dans l'eau, les résidus apolaires sont en majorité au centre et les
résidus chargés en surface : c'est la règle **« charges dehors, gras dedans »**. Un résidu enfoui
est entouré d'atomes de la protéine dans toutes les directions ; un résidu de surface n'en a que
d'un côté, l'autre est au contact de l'eau (les molécules d'eau ne sont pas dans la liste des
atomes).

On estime l'enfouissement d'un résidu par son nombre de **voisins** : le nombre d'atomes du
tétramère situés à moins de 10 Å d'au moins un atome de sa **chaîne latérale**. Dans un résidu,
les atomes `N`, `CA`, `C` et `O` forment le squelette, commun aux vingt acides aminés (liste
`SQUELETTE`, qui contient aussi `OXT`, l'oxygène terminal de la chaîne) ; tous les autres atomes
forment la chaîne latérale.

**Exercice.** Écrivez `voisins(atomes, chaine, residu, rayon=10)`. Les atomes du résidu lui-même ne
comptent pas, et chaque atome compte une seule fois, même s'il est proche de plusieurs atomes de
la chaîne latérale.

Appliquez-la au résidu 6 de la chaîne β normale (Glu6, dans 2HHB), au résidu 6 de la chaîne β du
patient (Val6, dans 2HBS) et, comme point de comparaison, à la valine 137 de la chaîne β
normale : parmi les valines de cette chaîne, celle qui a le plus de voisins. Même chaîne latérale
que Val6, autre position.

2HBS contient deux tétramères : `garder_chaines(hb_patient, "ABCD")` garde le premier.

In [ ]:
def voisins(atomes, chaine, residu, rayon=10):
    # 1. les atomes de la chaîne latérale : ceux du résidu dont le nom n'est pas dans SQUELETTE
    laterale = []
    for atome in atomes:
        if atome["chaine"] == chaine and atome["residu"] == residu and atome["atome"] not in ___:
            laterale.append(atome)
    # 2. les autres atomes, à moins de `rayon` Å d'au moins un atome de la chaîne latérale
    nombre = 0
    for atome in atomes:
        # un atome du résidu lui-même : on passe à l'atome suivant
        if atome["chaine"] == chaine and atome["residu"] == ___:
            continue
        for a in laterale:
            if distance(atome, a) < ___:
                # compté une seule fois : on sort de la boucle au premier atome proche
                nombre = nombre + 1
                break
    return nombre


# 2HBS contient deux tétramères : on garde le premier, chaînes A à D
tetramere_patient = garder_chaines(hb_patient, "ABCD")

print("Glu6, chaîne β normale (2HHB)    :", voisins(hb_normale, "B", 6), "voisins")
# le résidu 6 de la chaîne B, dans le tétramère du patient
print("Val6, chaîne β du patient (2HBS) :", voisins(___, "B", 6), "voisins")
print("Val137, chaîne β normale (2HHB)  :", voisins(hb_normale, "B", ___), "voisins")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
exemple = [
    nouvel_atome("A", 1, "N", -1, 0, 0),      # squelette du résidu 1 : ni compté, ni centre
    nouvel_atome("A", 1, "CB", 0, 0, 0),      # chaîne latérale du résidu 1
    nouvel_atome("A", 1, "CG", 2, 0, 0),      # chaîne latérale du résidu 1
    nouvel_atome("A", 2, "CA", 11, 0, 0),     # à 9 Å de CG : voisin
    nouvel_atome("A", 3, "CA", -10.5, 0, 0),  # à 10,5 Å de CB : trop loin
    nouvel_atome("B", 1, "CA", 0, 5, 0),      # autre chaîne : voisin, proche de CB et de CG
]
resultat = voisins(exemple, "A", 1)
if resultat == 2 and voisins(exemple, "A", 1, rayon=6) == 1:
    print("✅ sur un exemple de 6 atomes : 2 voisins à 10 Å, 1 à 6 Å")
else:
    print("❌ sur l'exemple : attendu 2 voisins, obtenu", resultat,
          "— le résidu lui-même est exclu, seuls les atomes hors SQUELETTE servent de centres,",
          "et un atome proche de plusieurs atomes de la chaîne latérale compte une seule fois")

resultats = [voisins(hb_normale, "B", 6), voisins(garder_chaines(hb_patient, "ABCD"), "B", 6),
             voisins(hb_normale, "B", 137)]
if resultats == [70, 73, 242]:
    print("✅ Glu6 :", resultats[0], "voisins ; Val6 :", resultats[1], "; Val137 :", resultats[2])
else:
    print("❌ attendu [70, 73, 242] voisins pour Glu6, Val6 et Val137, obtenu", resultats)

Pour situer ces nombres, la cellule suivante calcule les voisins de tous les résidus de la chaîne
β normale — sauf les glycines, qui n'ont pas de chaîne latérale — et les classe.

In [ ]:
# les résidus de la chaîne β normale (sauf les glycines, sans chaîne latérale), du moins
# entouré au plus entouré
classement = []
for numero, ca in carbones_alpha(hb_normale, "B").items():
    if ca["nom_residu"] != "GLY":
        classement.append((voisins(hb_normale, "B", numero), numero, ca["nom_residu"]))
classement.sort()

print("les 5 résidus qui ont le moins de voisins :")
for nombre, numero, nom in classement[:5]:
    print("   ", nom, numero, ":", nombre)
print("les 5 résidus qui en ont le plus :")
for nombre, numero, nom in classement[-5:]:
    print("   ", nom, numero, ":", nombre)

rang = 1
for nombre, numero, nom in classement:
    if numero == 6:
        break
    rang = rang + 1
print()
print("résidu 6 : rang", rang, "sur", len(classement), "en partant du moins entouré")

Dans la vue ci-dessous, la surface de chaque tétramère est colorée selon la classe de l'acide aminé
qui la forme : **apolaire** en orange, **polaire** en bleu clair, **acide** (chargé −) en rouge,
**basique** (chargé +) en bleu. À gauche l'hémoglobine normale (2HHB), à droite celle du patient
(2HBS, premier tétramère). Chaque vue place au centre, face à l'écran, le résidu 6 de la chaîne β
`B`, étiqueté ; faites tourner la molécule pour trouver celui de l'autre chaîne β, `D`.

In [ ]:
#@title ▶️ Vue 3D : la surface, colorée par classe d'acide aminé — normale (à gauche), patient (à droite)
vue = vue_3d([telecharger("2HHB.pdb"), telecharger("2HBS.pdb")], liees=False)
tetramere = {"chain": ["A", "B", "C", "D"], "hetflag": False}
vue.addSurface("VDW", {"colorscheme": {"prop": "resn", "map": COULEUR_RESIDU}}, tetramere)
vue.addLabel("Glu6 (B)", ETIQUETTE, {"chain": "B", "resi": 6}, viewer=(0, 0))
vue.addLabel("Val6 (B)", ETIQUETTE, {"chain": "B", "resi": 6}, viewer=(0, 1))
vue.zoomTo(tetramere)
# chaque vue tournée pour placer le résidu 6 de la chaîne B face à l'écran, au centre
tourner_vers(vue, hb_normale, "B", 6, 0)
tourner_vers(vue, garder_chaines(hb_patient, "ABCD"), "B", 6, 1)
vue.show()

Le résidu 6 est parmi les moins entourés de la chaîne β (rang 3 sur 133) : il est
**en surface**. Chez le patient, la tache rouge du glutamate 6 y est remplacée par une tache
orange : un groupe apolaire exposé à l'eau. D'après la règle « charges dehors, gras dedans »,
c'est un état coûteux : une surface apolaire au contact de l'eau s'associe à la première autre
surface apolaire accessible.

## 8. Pourquoi les tétramères s'accrochent

La structure `2HBS` contient **deux tétramères** d'HbS, dans la position qu'ils occupent l'un par
rapport à l'autre dans le cristal : chaînes A à D pour le premier, E à H pour le second (chaînes α :
A, C, E, G ; chaînes β : B, D, F, H).

**Exercice.** Écrivez `distance_min(atomes, residu1, residu2)`, où `residu1` et `residu2` sont des
tuples `(chaîne, numéro)` : la plus petite distance entre un atome du premier résidu et un atome du
second.

Appliquez-la à la valine 6 de la chaîne H (second tétramère) et à deux résidus apolaires de la
chaîne B (premier tétramère) : la phénylalanine 85 et la leucine 88.

In [ ]:
def distance_min(atomes, residu1, residu2):
    # chaque résidu est un tuple (chaîne, numéro)
    chaine1, numero1 = residu1
    chaine2, numero2 = ___
    # les atomes de chacun des deux résidus
    atomes1 = []
    atomes2 = []
    for atome in atomes:
        if atome["chaine"] == chaine1 and atome["residu"] == numero1:
            atomes1.append(atome)
        if atome["chaine"] == ___ and atome["residu"] == ___:
            atomes2.append(atome)
    # toutes les distances entre un atome du premier résidu et un atome du second
    distances = []
    for a1 in atomes1:
        for a2 in ___:
            distances.append(distance(a1, a2))
    # la plus petite : fonction min()
    return ___


print("Val6 (H) – Phe85 (B) :", round(distance_min(hb_patient, ("H", 6), ("B", 85)), 2), "Å")
# même chose avec la leucine 88 de la chaîne B
print("Val6 (H) – Leu88 (B) :", round(distance_min(hb_patient, ("H", 6), ___), 2), "Å")

In [ ]:
# ✔️ Vérification — exécutez sans modifier
exemple = [
    nouvel_atome("A", 1, "CB", 0, 0, 0),
    nouvel_atome("A", 1, "CG", 1, 0, 0),
    nouvel_atome("B", 2, "CB", 5, 0, 0),
    nouvel_atome("B", 2, "CG", 1, 0, 3),
    nouvel_atome("B", 3, "CB", 1, 0, 0.5),   # un autre résidu : ignoré
]
resultat = distance_min(exemple, ("A", 1), ("B", 2))
if proche(resultat, 3, 1e-9):
    print("✅ sur un exemple : 3.0 Å, entre CG (1, 0, 0) et CG (1, 0, 3)")
else:
    print("❌ sur l'exemple : attendu 3.0, obtenu", resultat)

d85 = distance_min(hb_patient, ("H", 6), ("B", 85))
d88 = distance_min(hb_patient, ("H", 6), ("B", 88))
if proche(d85, 3.95, 0.005) and proche(d88, 4.18, 0.005):
    print("✅ Val6 (H) : à", round(d85, 2), "Å de Phe85 (B) et à", round(d88, 2), "Å de Leu88 (B)")
else:
    print("❌ attendu 3.95 Å (Phe85) et 4.18 Å (Leu88), obtenu", d85, "et", d88)

La cellule suivante mesure, pour chacune des quatre valines 6 de la structure, la distance à la
poche Phe85 / Leu88 de chaque chaîne β, puis compte les voisins de la valine 6 de la chaîne H avec
et sans le tétramère voisin.

In [ ]:
print("distance minimale (Å) entre la Val6 d'une chaîne β et la poche Phe85/Leu88 d'une chaîne β")
print("               poche B   poche D   poche F   poche H")
for chaine_val in "BDFH":
    ligne = "Val6 de " + chaine_val + " :"
    for chaine_poche in "BDFH":
        d85 = distance_min(hb_patient, (chaine_val, 6), (chaine_poche, 85))
        d88 = distance_min(hb_patient, (chaine_val, 6), (chaine_poche, 88))
        ligne = ligne + f"{min(d85, d88):10.2f}"
    print(ligne)

print()
print("voisins de Val6 (H), dans son propre tétramère (E à H)   :",
      voisins(garder_chaines(hb_patient, "EFGH"), "H", 6))
print("voisins de Val6 (H), avec le tétramère voisin (A à H)    :", voisins(hb_patient, "H", 6))

Dans ce cristal, une seule des quatre valines 6 touche une poche : celle de la chaîne H, à
3,95 Å de Phe85 et à 4,18 Å de Leu88 de la chaîne B. C'est à peine plus que la somme des
rayons de van der Waals de deux atomes de carbone (1,7 + 1,7 = 3,4 Å) : la valine et la poche sont
au contact. Les trois autres valines 6 sont à plus de 17,9 Å de toute poche. Exposée dans son propre tétramère
(82 voisins), la valine 6 de la chaîne H est enfouie entre les deux tétramères :
229 voisins, au-delà de la médiane des résidus de la chaîne β normale (182).

Dans la vue ci-dessous : à gauche, les deux tétramères, le premier en bleu clair et le second en
gris, avec la valine 6 de la chaîne H et la poche Phe85 / Leu88 de la chaîne B en sphères
orange ; à droite, le contact agrandi : les mêmes trois résidus, apolaires, en bâtonnets orange.

In [ ]:
#@title ▶️ Vue 3D : deux tétramères de 2HBS (à gauche), le contact agrandi (à droite)
fichier = telecharger("2HBS.pdb")
vue = vue_3d([fichier, fichier], liees=False)
val6 = {"chain": "H", "resi": 6}
poche = {"chain": "B", "resi": [85, 88]}
apolaire = COULEUR_CLASSE["apolaire"]

# à gauche : premier tétramère (A à D) en bleu clair, second (E à H) en gris
vue.setStyle({"chain": ["A", "B", "C", "D"]}, {"cartoon": {"color": "#0ea5e9"}}, viewer=(0, 0))
vue.setStyle({"chain": ["E", "F", "G", "H"]}, {"cartoon": {"color": "#94a3b8"}}, viewer=(0, 0))
vue.addStyle(val6, {"sphere": {"color": apolaire}}, viewer=(0, 0))
vue.addStyle(poche, {"sphere": {"color": apolaire}}, viewer=(0, 0))
vue.zoomTo({"hetflag": False}, viewer=(0, 0))

# à droite : le contact agrandi, Val6 (H) et la poche Phe85/Leu88 (B) en bâtonnets
vue.setStyle({"chain": "B"}, {"cartoon": {"color": "#0ea5e9", "opacity": 0.6}}, viewer=(0, 1))
vue.setStyle({"chain": "H"}, {"cartoon": {"color": "#94a3b8", "opacity": 0.6}}, viewer=(0, 1))
vue.addStyle(val6, {"stick": {"color": apolaire, "radius": 0.3}}, viewer=(0, 1))
vue.addStyle(poche, {"stick": {"color": apolaire, "radius": 0.3}}, viewer=(0, 1))
vue.addLabel("Val6 (H)", ETIQUETTE, val6, viewer=(0, 1))
vue.addLabel("Phe85 (B)", ETIQUETTE, {"chain": "B", "resi": 85}, viewer=(0, 1))
vue.addLabel("Leu88 (B)", ETIQUETTE, {"chain": "B", "resi": 88}, viewer=(0, 1))
vue.zoomTo({"or": [val6, poche]}, viewer=(0, 1))
vue.zoom(0.6, viewer=(0, 1))
vue.show()

**Question.** Et si le résidu 6 était un glutamate, chargé, comme dans l'hémoglobine normale ?
Pourrait-il occuper la poche Phe85 / Leu88 ? Répondez avec la règle « charges dehors, gras
dedans ».

✍️ *Votre réponse (double-cliquez sur cette cellule pour écrire) :*

## Bilan

| Question | Mesure | Résultat |
|---|---|---|
| La prédiction est-elle fiable ? | pLDDT moyen, modèle AlphaFold DB de la chaîne β | 97,2 : très fiable |
| La chaîne β du patient a-t-elle le même repliement ? | dRMSD, chaîne β de 2HHB contre 2HBS | 0,26 Å, contre 0,19 Å entre deux copies de β et 1,12 Å entre α et β |
| Où est le résidu 6 ? | voisins à moins de 10 Å de la chaîne latérale | 70 (Glu6) et 73 (Val6), contre 242 pour Val137 : en surface |
| Qu'est-ce qui accroche deux tétramères ? | distance minimale dans 2HBS | Val6 (H) à 3,95 Å de Phe85 et 4,18 Å de Leu88 (B) |

**La chaîne β du patient se replie comme la chaîne normale ; ce qui change, c'est sa surface.**
Le résidu 6 est exposé à l'eau. La mutation y remplace un groupe chargé, le –CH₂–CH₂–COO⁻ du
glutamate, par un groupe apolaire, le –CH(CH₃)₂ de la valine : une tache apolaire à la surface de
chaque chaîne β. Cette tache s'emboîte dans une poche apolaire (Phe85, Leu88) d'un tétramère
voisin, à environ 4 Å. De tétramère en tétramère, l'HbS désoxygénée s'assemble ainsi en longues
fibres, qui rigidifient le globule rouge et le déforment en faucille.

**Le même glutamate, trois numérotations** (sections 3 et 4) :
- index Python **6** dans la chaîne traduite ;
- position **7** dans la chaîne traduite, méthionine comprise : c'est aussi la numérotation du
  modèle AlphaFold DB (UniProt) ;
- position **6** dans la protéine mature : littérature (Glu6Val), fichiers PDB, prédictions du
  serveur, et tout ce notebook (index Python 5 dans `beta_normale`).